In [ ]:
import pandas as pd

DATA = '../data/individual/processed'

baseline_eye_tracking = pd.read_csv(f'{DATA}/sed.csv')
baseline_eye_tracking = baseline_eye_tracking.rename(columns={
    'datetime': 'timestamp',
    'pupil': 'pupil_dilation',
    'leftEyeOpen': 'left_blink',
    'rightEyeOpen': 'right_blink'
})

baseline_eye_tracking['timestamp'] = pd.to_datetime(baseline_eye_tracking['timestamp'], utc=True, errors='coerce').dt.tz_convert(None)
baseline_eye_tracking = baseline_eye_tracking.dropna(subset=['timestamp'])

start_time = baseline_eye_tracking['timestamp'].min()
end_time = baseline_eye_tracking['timestamp'].max()
total_duration_minutes = (end_time - start_time).total_seconds() / 60

average_pupil_dilation = baseline_eye_tracking['pupil_dilation'].mean()

BLINK_THRESHOLD = 1.0
MIN_CLOSED_FRAMES = 3  # ~100ms at 30Hz

def count_blinks(series, threshold, min_frames):
    # sustained closure onset
    closed = (series <= threshold).astype(int)
    sustained = closed.rolling(min_frames).sum() == min_frames
    return int((sustained & ~sustained.shift(1, fill_value=False)).sum())

left_blink_count = count_blinks(baseline_eye_tracking['left_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES)
right_blink_count = count_blinks(baseline_eye_tracking['right_blink'], BLINK_THRESHOLD, MIN_CLOSED_FRAMES)

if total_duration_minutes <= 0:
    left_blink_rate, right_blink_rate = 0.0, 0.0
else:
    left_blink_rate = left_blink_count / total_duration_minutes
    right_blink_rate = right_blink_count / total_duration_minutes

print(f"Baseline Metrics:\n")
print(f"  Start Time: {start_time}")
print(f"  End Time: {end_time}")
print(f"  Total Duration: {total_duration_minutes:.2f} minutes")
print(f"  Average Pupil Dilation: {average_pupil_dilation:.2f}")
print(f"  Left Blink Rate: {left_blink_rate:.2f} blinks/min")
print(f"  Right Blink Rate: {right_blink_rate:.2f} blinks/min")
